In [ ]:

# Titanic Exploratory Data Analysis (EDA)
# Student Name: Shrey Shah
# Course: Data Analysis
# Libraries: NumPy, Pandas, Matplotlib, Seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


class TitanicEDA:

    def __init__(self):
        self.train = pd.DataFrame()
        self.test = pd.DataFrame()
        self.submission = pd.DataFrame()
        self.output_folder = "outputs"

        try:
            os.makedirs(self.output_folder, exist_ok=True)
        except OSError as e:
            print("Unable to create output folder:", e)

    # 1. Load CSV files
    def load_data(self):
        try:
            self.train = pd.read_csv("train.csv")
            self.test = pd.read_csv("test.csv")
            self.submission = pd.read_csv("gender_submission.csv")

            print("\nCSV files loaded successfully.")
            print("Training data shape:", self.train.shape)
            print("Testing data shape:", self.test.shape)
            print("Submission data shape:", self.submission.shape)

            # Automatically save report as a text file
            self.generate_text_report()

        except FileNotFoundError as e:
            print("File not found:", e)
            print("Keep all CSV files in the project folder.")

        except pd.errors.EmptyDataError:
            print("One of the CSV files is empty.")

        except pd.errors.ParserError as e:
            print("CSV parsing error:", e)

        except (OSError, ValueError) as e:
            print("Unable to load data:", e)

    # Check whether data is loaded
    def check_data(self):
        if self.train.empty:
            print("Please load the CSV files first.")
            return False
        return True

    # 2. Explore dataset
    def explore_data(self):
        if not self.check_data():
            return

        print("\nFIRST FIVE ROWS")
        print(self.train.head())

        print("\nLAST FIVE ROWS")
        print(self.train.tail())

        print("\nCOLUMN NAMES")
        print(self.train.columns.tolist())

        print("\nDATA TYPES")
        print(self.train.dtypes)

        print("\nDATASET INFORMATION")
        self.train.info()

        print("\nDESCRIPTIVE STATISTICS")
        print(self.train.describe(include="all").T)

        print("\nDATASET DIMENSIONS")
        print("Rows:", self.train.shape[0])
        print("Columns:", self.train.shape[1])

    # 3. Missing values and duplicate records
    def missing_values(self):
        if not self.check_data():
            return

        print("\nMISSING VALUES IN TRAIN DATA")
        print(self.train.isnull().sum())

        print("\nMISSING VALUES IN TEST DATA")
        if not self.test.empty:
            print(self.test.isnull().sum())

        print("\nDUPLICATE ROWS")
        print(self.train.duplicated().sum())

        print("\nMISSING VALUE PERCENTAGE")
        missing_percent = (
            self.train.isnull().mean() * 100
        ).round(2)

        print(missing_percent[missing_percent > 0])

    # 4. Clean training data
    def clean_data(self):
        if not self.check_data():
            return

        try:
            cleaned = self.train.copy()

            cleaned = cleaned.drop_duplicates()

            # Fill missing age values with median
            if "Age" in cleaned.columns:
                cleaned["Age"] = cleaned["Age"].fillna(
                    cleaned["Age"].median()
                )

            # Fill missing embarkation values with mode
            if "Embarked" in cleaned.columns:
                mode_value = cleaned["Embarked"].mode()

                if not mode_value.empty:
                    cleaned["Embarked"] = cleaned["Embarked"].fillna(
                        mode_value.iloc[0]
                    )

            # Create a cabin availability indicator
            if "Cabin" in cleaned.columns:
                cleaned["HasCabin"] = (
                    cleaned["Cabin"].notna().astype(int)
                )

            file_path = os.path.join(
                self.output_folder, "cleaned_train.csv"
            )

            cleaned.to_csv(file_path, index=False)

            print("\nData cleaning completed.")
            print("Original rows:", len(self.train))
            print("Cleaned rows:", len(cleaned))
            print("Cleaned file saved to:", file_path)

            print("\nRemaining missing values:")
            print(cleaned.isnull().sum())

        except (KeyError, TypeError, ValueError, OSError) as e:
            print("Error while cleaning data:", e)

    # 5. Survival analysis
    def survival_analysis(self):
        if not self.check_data():
            return

        try:
            print("\nSURVIVAL COUNTS")
            print(self.train["Survived"].value_counts())

            print("\nSURVIVAL PERCENTAGES")
            print(
                (
                    self.train["Survived"]
                    .value_counts(normalize=True) * 100
                ).round(2)
            )

            print("\nSURVIVAL RATE BY GENDER (%)")
            print(
                self.train.groupby("Sex")["Survived"]
                .mean().mul(100).round(2)
            )

            print("\nSURVIVAL RATE BY PASSENGER CLASS (%)")
            print(
                self.train.groupby("Pclass")["Survived"]
                .mean().mul(100).round(2)
            )

            print("\nSURVIVAL RATE BY EMBARKATION PORT (%)")
            print(
                self.train.groupby("Embarked")["Survived"]
                .mean().mul(100).round(2)
            )

            print("\nAVERAGE AGE BY SURVIVAL")
            print(
                self.train.groupby("Survived")["Age"]
                .mean().round(2)
            )

            print("\nAVERAGE FARE BY SURVIVAL")
            print(
                self.train.groupby("Survived")["Fare"]
                .mean().round(2)
            )

        except (KeyError, TypeError, ValueError) as e:
            print("Error during survival analysis:", e)

    # 6. NumPy analysis
    def numpy_analysis(self):
        if not self.check_data():
            return

        try:
            ages = self.train["Age"].dropna().to_numpy()
            fares = self.train["Fare"].dropna().to_numpy()

            if len(ages) > 0:
                print("\nNUMPY AGE ANALYSIS")
                print("Mean age:", round(np.mean(ages), 2))
                print("Median age:", np.median(ages))
                print("Minimum age:", np.min(ages))
                print("Maximum age:", np.max(ages))
                print("Standard deviation:", round(np.std(ages), 2))

            if len(fares) > 0:
                print("\nNUMPY FARE ANALYSIS")
                print("Mean fare:", round(np.mean(fares), 2))
                print("Median fare:", np.median(fares))
                print("Minimum fare:", np.min(fares))
                print("Maximum fare:", np.max(fares))

            if "Survived" in self.train.columns:
                survival = self.train["Survived"].to_numpy()
                print(
                    "\nOverall survival rate (%):",
                    round(np.mean(survival) * 100, 2)
                )

        except (KeyError, TypeError, ValueError) as e:
            print("NumPy analysis error:", e)

    # 7. Generate visualizations
    def visualize(self):
        if not self.check_data():
            return

        try:
            sns.set_theme(style="whitegrid")

            # Chart 1: Survival count
            plt.figure(figsize=(7, 5))
            sns.countplot(data=self.train, x="Survived")
            plt.title("Titanic Passenger Survival Count")
            plt.xlabel("Survived (0 = No, 1 = Yes)")
            plt.ylabel("Number of Passengers")
            plt.tight_layout()
            self.save_chart("01_survival_count.png")

            # Chart 2: Survival by gender
            plt.figure(figsize=(7, 5))
            sns.countplot(
                data=self.train,
                x="Sex",
                hue="Survived"
            )
            plt.title("Survival by Gender")
            plt.xlabel("Gender")
            plt.ylabel("Number of Passengers")
            plt.tight_layout()
            self.save_chart("02_survival_by_gender.png")

            # Chart 3: Survival rate by passenger class
            plt.figure(figsize=(7, 5))
            sns.barplot(
                data=self.train,
                x="Pclass",
                y="Survived",
                errorbar=None
            )
            plt.title("Survival Rate by Passenger Class")
            plt.xlabel("Passenger Class")
            plt.ylabel("Survival Rate")
            plt.tight_layout()
            self.save_chart("03_survival_by_class.png")

            # Chart 4: Age distribution
            plt.figure(figsize=(8, 5))
            sns.histplot(
                data=self.train,
                x="Age",
                bins=25,
                kde=True
            )
            plt.title("Distribution of Passenger Ages")
            plt.xlabel("Age")
            plt.ylabel("Frequency")
            plt.tight_layout()
            self.save_chart("04_age_distribution.png")

            # Chart 5: Fare distribution
            plt.figure(figsize=(8, 5))
            sns.histplot(
                data=self.train,
                x="Fare",
                bins=30,
                kde=True
            )
            plt.title("Distribution of Ticket Fares")
            plt.xlabel("Fare")
            plt.ylabel("Frequency")
            plt.tight_layout()
            self.save_chart("05_fare_distribution.png")

            # Chart 6: Age versus survival
            plt.figure(figsize=(8, 5))
            sns.boxplot(
                data=self.train,
                x="Survived",
                y="Age"
            )
            plt.title("Age Distribution by Survival")
            plt.xlabel("Survived (0 = No, 1 = Yes)")
            plt.ylabel("Age")
            plt.tight_layout()
            self.save_chart("06_age_vs_survival.png")

            # Chart 7: Correlation heatmap
            numeric_data = self.train.select_dtypes(
                include=np.number
            )

            plt.figure(figsize=(10, 7))
            sns.heatmap(
                numeric_data.corr(),
                annot=True,
                cmap="coolwarm",
                fmt=".2f",
                linewidths=0.5
            )
            plt.title("Correlation Between Numerical Variables")
            plt.tight_layout()
            self.save_chart("07_correlation_heatmap.png")

            print("\nAll visualizations generated.")

        except (KeyError, TypeError, ValueError, OSError) as e:
            print("Visualization error:", e)

    # 8. Save chart
    def save_chart(self, filename):
        try:
            file_path = os.path.join(
                self.output_folder, filename
            )

            plt.savefig(
                file_path,
                dpi=300,
                bbox_inches="tight"
            )

            print("Chart saved:", file_path)
            plt.show()
            plt.close()

        except (OSError, ValueError) as e:
            print("Unable to save chart:", e)
            plt.close()

    # 9. Inspect test and submission data
    def inspect_test_data(self):
        if self.test.empty:
            print("Test data is not loaded.")
            return

        if self.submission.empty:
            print("Submission data is not loaded.")
            return

        try:
            print("\nTEST DATA PREVIEW")
            print(self.test.head())

            print("\nMISSING VALUES IN TEST DATA")
            print(self.test.isnull().sum())

            print("\nGENDER SUBMISSION PREVIEW")
            print(self.submission.head())

            print("\nSUBMISSION SURVIVAL COUNTS")
            print(self.submission["Survived"].value_counts())

            if (
                "PassengerId" in self.test.columns
                and "PassengerId" in self.submission.columns
            ):
                same_ids = self.test["PassengerId"].reset_index(
                    drop=True
                ).equals(
                    self.submission["PassengerId"].reset_index(
                        drop=True
                    )
                )

                print(
                    "\nSubmission PassengerIds match test order:",
                    same_ids
                )

        except (KeyError, TypeError, ValueError) as e:
            print("Error inspecting test data:", e)

    # 10. Generate and save the text report
    def generate_text_report(self):
        if not self.check_data():
            return

        try:
            report_path = os.path.join(
                self.output_folder, "eda_report.txt"
            )

            with open(report_path, "w", encoding="utf-8") as file:

                file.write(
                    "TITANIC EXPLORATORY DATA ANALYSIS REPORT\n"
                )
                file.write("=" * 45 + "\n\n")

                file.write("STUDENT NAME: Shrey Shah\n")
                file.write("COURSE: Data Analysis\n\n")

                file.write("1. DATASET DIMENSIONS\n")
                file.write(f"Rows: {self.train.shape[0]}\n")
                file.write(f"Columns: {self.train.shape[1]}\n\n")

                file.write("2. COLUMN DATA TYPES\n")
                file.write(self.train.dtypes.to_string())
                file.write("\n\n")

                file.write("3. MISSING VALUES\n")
                file.write(self.train.isnull().sum().to_string())
                file.write("\n\n")

                file.write("4. DUPLICATE ROWS\n")
                file.write(str(self.train.duplicated().sum()))
                file.write("\n\n")

                file.write("5. DESCRIPTIVE STATISTICS\n")
                file.write(
                    self.train.describe().round(2).to_string()
                )
                file.write("\n\n")

                file.write("6. SURVIVAL COUNTS\n")
                file.write(
                    self.train["Survived"].value_counts().to_string()
                )
                file.write("\n\n")

                file.write("7. SURVIVAL RATE BY GENDER (%)\n")
                file.write(
                    (
                        self.train.groupby("Sex")["Survived"]
                        .mean().mul(100).round(2)
                    ).to_string()
                )
                file.write("\n\n")

                file.write("8. SURVIVAL RATE BY CLASS (%)\n")
                file.write(
                    (
                        self.train.groupby("Pclass")["Survived"]
                        .mean().mul(100).round(2)
                    ).to_string()
                )
                file.write("\n\n")

                file.write("9. NUMPY ANALYSIS\n")

                for column in ["Age", "Fare"]:
                    if column in self.train.columns:
                        values = (
                            self.train[column].dropna().to_numpy()
                        )

                        if len(values) > 0:
                            file.write(f"\n{column}\n")
                            file.write(
                                f"Mean: {np.mean(values):.2f}\n"
                            )
                            file.write(
                                f"Median: {np.median(values):.2f}\n"
                            )
                            file.write(
                                f"Standard deviation: "
                                f"{np.std(values):.2f}\n"
                            )

                file.write("\nEND OF REPORT\n")

            print("\nReport saved successfully to:")
            print(os.path.abspath(report_path))

        except (OSError, KeyError, TypeError, ValueError) as e:
            print("Error generating text report:", e)

    # 11. Menu-driven program
    def menu(self):
        while True:
            print("\n" + "=" * 40)
            print("TITANIC EDA PROJECT")
            print("=" * 40)
            print("1. Load CSV files and generate text report")
            print("2. Explore dataset")
            print("3. Check missing values and duplicates")
            print("4. Clean training data")
            print("5. Perform survival analysis")
            print("6. Perform NumPy analysis")
            print("7. Generate visualizations")
            print("8. Inspect test and submission data")
            print("9. Exit")

            try:
                choice = int(input("Enter your choice: "))

                if choice == 1:
                    self.load_data()

                elif choice == 2:
                    self.explore_data()

                elif choice == 3:
                    self.missing_values()

                elif choice == 4:
                    self.clean_data()

                elif choice == 5:
                    self.survival_analysis()

                elif choice == 6:
                    self.numpy_analysis()

                elif choice == 7:
                    self.visualize()

                elif choice == 8:
                    self.inspect_test_data()

                elif choice == 9:
                    print("Thank you for using Titanic EDA!")
                    break

                else:
                    print("Invalid choice. Select from 1 to 9.")

            except ValueError:
                print("Please enter a valid numeric choice.")

            except (KeyboardInterrupt, EOFError):
                print("\nProgram interrupted.")
                break


# Main program
if __name__ == "__main__":
    try:
        app = TitanicEDA()
        app.menu()

    except Exception as e:
        print("Unexpected program error:", e)
